# SQL Assignment: Database Schema & Querying

---

## Part 1: Conceptual Questions

### Question 1: SQL Commands (DDL vs. DML vs. DQL)
*   **DDL (Data Definition Language):** Defines, alters, or drops structural schema objects (databases, tables, views, indexes).
    *   *Properties:* Auto-committed, modifications are permanent.
    *   *Example:* `CREATE TABLE`
*   **DML (Data Manipulation Language):** Handles inserting, updating, or deleting actual records within existing tables.
    *   *Properties:* Supports transaction control (can be rolled back).
    *   *Example:* `INSERT INTO`
*   **DQL (Data Query Language):** Specifically retrieves data records without altering schemas or contents.
    *   *Properties:* Does not modify state.
    *   *Example:* `SELECT`

---

### Question 2: Constraints in SQL
*   **Purpose:** Ensures the accuracy, integrity, validation, and safety of data written to tables.
*   **Common Constraints:**
    1.  **PRIMARY KEY:** Uniquely identifies each row, enforcing uniqueness and non-nullability (e.g., unique `CustomerID` for records).
    2.  **FOREIGN KEY:** Guarantees referential integrity by linking a column to a primary key in another table (e.g., linking product category reference).
    3.  **UNIQUE:** Restricts a column to only hold distinct non-duplicate values (e.g., preventing duplicate customer Emails).

---

### Question 3: Pagination with LIMIT and OFFSET
*   `LIMIT`: Limits the max number of output rows.
*   `OFFSET`: Skips a specific quantity of rows.
*   *Usage for page 3 (10 per page):* Skip 2 pages (20 items) and return next 10 items.

```sql
SELECT *
FROM Products
ORDER BY ProductID
LIMIT 10 OFFSET 20;
```

---

### Question 4: Common Table Expressions (CTEs)
*   **Definition:** Temporary named result sets defined using the `WITH` keyword for readability and breakdown of complex queries.

```sql
WITH ExpensiveProducts AS (
    SELECT ProductName, Price
    FROM Products
    WHERE Price > 100.00
)
SELECT ProductName, Price
FROM ExpensiveProducts
ORDER BY Price DESC;
```

---

### Question 5: Normalization (1NF, 2NF, 3NF)
*   **Goal:** Minimize storage redundancy, eliminate insertion/update anomalies, and secure relationships.
*   *1NF:* Every field value is atomic; distinct primary keys defined.
*   *2NF:* Must be 1NF; removes partial dependencies (no non-key attribute relies on part of a composite primary key).
*   *3NF:* Must be 2NF; removes transitive dependencies (no non-key column depends on another non-key column).

In [ ]:
#Question 6: Database Setup & DDL Schema
sql_schema_ddl = """
CREATE DATABASE IF NOT EXISTS ECommerceDB;
USE ECommerceDB;

CREATE TABLE IF NOT EXISTS Categories (
    CategoryID INT PRIMARY KEY,
    CategoryName VARCHAR(50) NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS Products (
    ProductID INT PRIMARY KEY,
    ProductName VARCHAR(100) NOT NULL UNIQUE,
    CategoryID INT,
    Price DECIMAL(10,2) NOT NULL,
    StockQuantity INT,
    FOREIGN KEY (CategoryID) REFERENCES Categories(CategoryID)
);

CREATE TABLE IF NOT EXISTS Customers (
    CustomerID INT PRIMARY KEY,
    CustomerName VARCHAR(100) NOT NULL,
    Email VARCHAR(100) UNIQUE,
    JoiningDate DATE
);

CREATE TABLE IF NOT EXISTS Orders (
    OrderID INT PRIMARY KEY,
    CustomerID INT,
    OrderDate DATE NOT NULL,
    TotalAmount DECIMAL(10,2),
    FOREIGN KEY (CustomerID) REFERENCES Customers(CustomerID)
);
"""
print("Schema definition structured.")

In [3]:
#@title DML Statements to Insert Records
sql_dml_insert = """
INSERT INTO Categories (CategoryID, CategoryName) VALUES
(1, 'Electronics'), (2, 'Books'), (3, 'Home Goods'), (4, 'Apparel');

INSERT INTO Products (ProductID, ProductName, CategoryID, Price, StockQuantity) VALUES
(101, 'Laptop Pro', 1, 1200.00, 50),
(102, 'SQL Handbook', 2, 45.50, 200),
(103, 'Smart Speaker', 1, 99.99, 150),
(104, 'Coffee Maker', 3, 75.00, 80),
(105, 'Novel: The Great SQL', 2, 25.00, 120),
(106, 'Wireless Earbuds', 1, 150.00, 100),
(107, 'Blender X', 3, 120.00, 60),
(108, 'T-Shirt Casual', 4, 20.00, 300);

INSERT INTO Customers (CustomerID, CustomerName, Email, JoiningDate) VALUES
(1, 'Alice Wonderland', 'alice@example.com', '2023-01-10'),
(2, 'Bob the Builder', 'bob@example.com', '2022-11-25'),
(3, 'Charlie Chaplin', 'charlie@example.com', '2023-03-01'),
(4, 'Diana Prince', 'diana@example.com', '2021-04-26');

INSERT INTO Orders (OrderID, CustomerID, OrderDate, TotalAmount) VALUES
(1001, 1, '2023-04-26', 1245.50),
(1002, 2, '2023-10-12', 99.99),
(1003, 1, '2023-07-01', 145.00),
(1004, 3, '2023-01-14', 150.00),
(1005, 2, '2023-09-24', 120.00),
(1006, 1, '2023-06-19', 20.00);
"""
print("DML mock data injection script ready.")

DML mock data injection script ready.


In [4]:
#@title Question 7: Customer Order Volume Report Query
sql_q7 = """
SELECT
    c.CustomerName,
    c.Email,
    COUNT(o.OrderID) AS TotalNumberofOrders
FROM
    Customers c
LEFT JOIN
    Orders o ON c.CustomerID = o.CustomerID
GROUP BY
    c.CustomerID,
    c.CustomerName,
    c.Email
ORDER BY
    c.CustomerName ASC;
"""

In [5]:
#@title Question 8: Catalog Breakdown Query
sql_q8 = """
SELECT
    p.ProductName,
    p.Price,
    p.StockQuantity,
    c.CategoryName
FROM
    Products p
JOIN
    Categories c ON p.CategoryID = c.CategoryID
ORDER BY
    c.CategoryName ASC,
    p.ProductName ASC;
"""

In [6]:
#@title Question 9: Top-2 Price Density per Category Query
sql_q9 = """
WITH RankedProducts AS (
    SELECT
        c.CategoryName,
        p.ProductName,
        p.Price,
        ROW_NUMBER() OVER (PARTITION BY c.CategoryName ORDER BY p.Price DESC) AS rn
    FROM
        Products p
    JOIN
        Categories c ON p.CategoryID = c.CategoryID
)
SELECT
    CategoryName,
    ProductName,
    Price
FROM
    RankedProducts
WHERE
    rn <= 2;
"""

## Part 3: Sakila Database Analytical Queries

In [8]:
#@title Sakila Analytical Tasks Queries

# 1. Top 5 Customers Spend
sakila_task_1 = """
SELECT
    CONCAT(c.first_name, ' ', c.last_name) AS CustomerName,
    c.email,
    SUM(p.amount) AS TotalSpent
FROM
    customer c
JOIN
    payment p ON c.customer_id = p.customer_id
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name,
    c.email
ORDER BY
    TotalSpent DESC
LIMIT 5;
"""

# 2. Top 3 Movie Categories by Rentals
sakila_task_2 = """
SELECT
    cat.name AS CategoryName,
    COUNT(r.rental_id) AS RentalCount
FROM
    category cat
JOIN
    film_category fc ON cat.category_id = fc.category_id
JOIN
    inventory i ON fc.film_id = i.film_id
JOIN
    rental r ON i.inventory_id = r.inventory_id
GROUP BY
    cat.category_id,
    cat.name
ORDER BY
    RentalCount DESC
LIMIT 3;
"""

# 3. Available Inventory vs Never Rented
sakila_task_3 = """
SELECT
    i.store_id,
    COUNT(i.inventory_id) AS TotalFilmsAvailable,
    SUM(CASE WHEN r.rental_id IS NULL THEN 1 ELSE 0 END) AS NeverRentedCount
FROM
    inventory i
LEFT JOIN
    rental r ON i.inventory_id = r.inventory_id
GROUP BY
    i.store_id;
"""

# 4. Monthly Seasonality (2023 Revenue)
sakila_task_4 = """
SELECT
    DATE_FORMAT(payment_date, '%Y-%m') AS RentalMonth,
    SUM(amount) AS TotalRevenue
FROM
    payment
WHERE
    YEAR(payment_date) = 2023
GROUP BY
    DATE_FORMAT(payment_date, '%Y-%m')
ORDER BY
    RentalMonth ASC;
"""

# 5. Customers with > 10 Rentals (Last 6 Months)
sakila_task_5 = """
SELECT
    c.customer_id,
    CONCAT(c.first_name, ' ', c.last_name) AS CustomerName,
    c.email,
    COUNT(r.rental_id) AS RentalCount
FROM
    customer c
JOIN
    rental r ON c.customer_id = r.customer_id
WHERE
    r.rental_date >= (SELECT DATE_SUB(MAX(rental_date), INTERVAL 6 MONTH) FROM rental)
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name,
    c.email
HAVING
    COUNT(r.rental_id) > 10
ORDER BY
    RentalCount DESC;
"""